In [48]:
import numpy as np
import pandas as pd
import geopandas as gpd
from path_config import PathConfig
paths = PathConfig()
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import math

In [ ]:
buildings = gpd.read_file(paths.build_path)
vegetation = gpd.read_file(paths.vege_path)
water = gpd.read_file(paths.waters_path)
roads = gpd.read_file(paths.roads_path)
border = gpd.read_file(paths.border_path)

layers = [buildings, vegetation, water, roads, border]
for layer in layers:
    if layer.crs != "EPSG:32635":
        layer = layer.to_crs(epsg=32635)

utm_crs = "EPSG:32635"

buildings, vegetation, water, roads, border = layers

base_stations_data = [
    {
        "lat": 41.1073413, "lon": 29.0233668, "azimuth": 50, 
        "scenario": "Macro", "height": 10,
        "hbw": 65.5, "vbw": 10, "downtilt": 8, 
        "v_pointing_error": 1, "h_pointing_error": 0.5
    },
    {
        "lat": 41.1070873, "lon": 29.0226646, "azimuth": 260, 
        "scenario": "Macro", "height": 10,
        "hbw": 65.5, "vbw": 10, "downtilt": 8,
        "v_pointing_error": 1, "h_pointing_error": 0.5
    },
    {
        "lat": 41.1080861, "lon": 29.0281222, "azimuth": 230, 
        "scenario": "Hotspot", "height": 6,
        "hbw": 65.5, "vbw": 30, "downtilt": 3,
        "v_pointing_error": 3, "h_pointing_error": 0.5
    },
    {
        "lat": 41.1080861, "lon": 29.0281222, "azimuth": 110, 
        "scenario": "Hotspot", "height": 6,
        "hbw": 65.5, "vbw": 30, "downtilt": 3,
        "v_pointing_error": 3, "h_pointing_error": 0.5
    },
    {
        "lat": 41.1054694, "lon": 29.0278333, "azimuth": 255, 
        "scenario": "Highrise", "height": 18,
        "hbw": 20, "vbw": 30, "downtilt": 3,
        "v_pointing_error": 3, "h_pointing_error": 0.2
    },
    {
        "lat": 41.1054694, "lon": 29.0278333, "azimuth": 340, 
        "scenario": "Highrise", "height": 18,
        "hbw": 20, "vbw": 30, "downtilt": 3,
        "v_pointing_error": 3, "h_pointing_error": 0.2
    },
    {
        "lat": 41.1043417, "lon": 29.0217472, "azimuth": 300, 
        "scenario": "Hotspot", "height": 6,
        "hbw": 65.5, "vbw": 30, "downtilt": 3,
        "v_pointing_error": 3, "h_pointing_error": 0.5
    },
    {
        "lat": 41.1043417, "lon": 29.0217472, "azimuth": 40, 
        "scenario": "Hotspot", "height": 6,
        "hbw": 65.5, "vbw": 30, "downtilt": 3,
        "v_pointing_error": 3, "h_pointing_error": 0.5
    },
    {
        "lat": 41.1058417, "lon": 29.0205028, "azimuth": 130, 
        "scenario": "Hotspot", "height": 6,
        "hbw": 65.5, "vbw": 30, "downtilt": 3,
        "v_pointing_error": 3, "h_pointing_error": 0.5
    },
]

stations_df = pd.DataFrame(base_stations_data)
stations_gdf = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(stations_df["lon"], stations_df["lat"]),
    crs="EPSG:4326"
).to_crs(utm_crs)


grid_spacing = 20 # spacing i şimdilik 20 yapıyorum görsel kasmasın diye, 5m ye düşürebiliriz bence.
border = border.to_crs(utm_crs)
minx, miny, maxx, maxy = border.total_bounds
x_coords = np.arange(minx, maxx, grid_spacing)
y_coords = np.arange(miny, maxy, grid_spacing)

grid_points = [Point(x, y) for x in x_coords for y in y_coords]
grid_gdf = gpd.GeoDataFrame(geometry=grid_points, crs=utm_crs).to_crs(utm_crs)

border_polygon = Polygon(border.geometry.iloc[0])
grid_gdf = grid_gdf[grid_gdf.geometry.within(border_polygon)]

grid_gdf.explore()